# Data Validation
The objective of this stage is to transform the raw datasets into a standardized, comparable format. Before performing the actual reconciliation (comparison), we must ensure that both files are technically consistent and free from structural "noise."

This step include: 
- missing data verification - identifying gaps (nulls) in critical columns like `UnitPrice` or `CustomerID`. We assess their impact and decide whether to impute, flag, or exclude them from the financial comparison.
- data type standardization - ensuring that key columns used for matching (e.g., `CustomerID`, `InvoiceNo`) have consistent data types in both datasets (converting strings to numeric or vice versa) to avoid join errors.
- formatting standardization - cleaning technical "blockers" (hidden spaces, unifying capitalziation, fixing corrupted data formats)
- duplicate detection and isolation - identifying redundant records that could distort totals. Technical duplicates will be moved to separate dataframes for audit purpose, while the main datasets are deduplicated to establish a "Single Source of Truth".

Purpose: To reach a state where both datasets are technically "aligned," allowing for an accurate row-by-row comparison in the Comparison phase.

**Step 1** Copy of both datasets.

In [4]:
import pandas as pd
import os

#folder for copied files
os.makedirs('validation', exist_ok=True)

def load_data(file_path):
    try:
        df = pd.read_csv(file_path, encoding='latin1')
        print(f"File {file_path} loaded correctly using latin1 encoding. Loaded {len(df)} rows.")
        return df
    except Exception:
        df = pd.read_csv(file_path, encoding='cp1252')
        print(f"File {file_path} loaded correctly using cp1252 encoding. Loaded {len(df)} rows.")
        return df

# oryginal files
df_oms_org = load_data('oms_orders_placed.csv')
df_wms_org = load_data('wms_shipped_units.csv')

# saving and loadin copy of files 
df_oms_org.to_csv('validation/oms_orders_placed_copy.csv', index=False, encoding='latin1')
df_wms_org.to_csv('validation/wms_shipped_units_copy.csv', index=False, encoding='latin1')

df_oms = pd.read_csv('validation/oms_orders_placed_copy.csv', encoding='latin1')
df_wms = pd.read_csv('validation/wms_shipped_units_copy.csv', encoding='latin1')

File oms_orders_placed.csv loaded correctly using latin1 encoding. Loaded 541909 rows.
File wms_shipped_units.csv loaded correctly using latin1 encoding. Loaded 542409 rows.


**Step 2** Missing data verification

In [7]:
def check_missing_data(df, name):
    print(f"Missing Data Analysis: {name}")
    missing = df.isnull().sum()
    missing_pct = (df.isnull().sum()/len(df)) * 100
    
    missing_df = pd.DataFrame({
        'Missing values': missing,
        'Percentage (%)': missing_pct
    })
    
    print(missing_df[missing_df['Missing values'] > 0])
    return missing_df

#running verification for both files
oms_missing = check_missing_data(df_oms, "OMS")
wms_missing = check_missing_data(df_wms, "WMS")


Missing Data Analysis: OMS
             Missing values  Percentage (%)
Description            1454        0.268311
CustomerID           135080       24.926694
Missing Data Analysis: WMS
             Missing values  Percentage (%)
Description            1454        0.268063
UnitPrice              1000        0.184363
CustomerID           134660       24.826284


Observation: 
- both datasets have same amount of missing values in Description
- both datasets are missing CustomerID, but in 'original' OMS the value is higher
- WMS dataset is missing 1000 UnitPrice

**Step 3** Data type verification adn standarization

In [8]:
df_oms.info()
df_wms.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 542409 entries, 0 to 542408
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    542409 non-null  object 
 1   StockCode    542409 non-null  object 
 2   Description  540955 non-null  object 
 3   Quantity     542409 non-null  int64  
 4   InvoiceDate  542409 non-

In [10]:
df_oms['InvoiceDate'] = pd.to_datetime(df_oms['InvoiceDate'], errors = 'coerce')
df_wms['InvoiceDate'] = pd.to_datetime(df_wms['InvoiceDate'], errors = 'coerce')

print(f"Incorrect dates in OMS: {df_oms['InvoiceDate'].isna().sum()}")
print(f"Incorrect dates in WMS: {df_wms['InvoiceDate'].isna().sum()}")

C:\Users\marty\AppData\Local\Temp\ipykernel_22896\3808704283.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_wms['InvoiceDate'] = pd.to_datetime(df_wms['InvoiceDate'], errors = 'coerce')


Incorrect dates in OMS: 0
Incorrect dates in WMS: 11


Observation:
- 11 incorrect dates in WMS identified. I'll move this 11 records to another file.

In [15]:
#create a variable with incorrect records (for audit)

wms_date_errors = df_wms[df_wms['InvoiceDate'].isna()].copy()

#saving into valdiation folder in separated file
wms_date_errors.to_csv('validation/wms_date_errors_audit.csv', index = False)

#removing from main WMS file that we use in valdiation process

df_wms = df_wms[df_wms['InvoiceDate'].notna()].copy()

print(f"Moved {len(wms_date_errors)} incorrect dates into audit file.")

Moved 11 incorrect dates into audit file.


**Step 4** Data type and formatting standarization

In [20]:
def clean_and_standardize(df):
    # use pd.to_numeric first to handle strings like '13047.0' then convert to Int to remove decimals, then to String for matching.
    df['CustomerID'] = pd.to_numeric(df['CustomerID'], errors='coerce') \
                        .fillna(0) \
                        .astype(int) \
                        .astype(str) \
                        .replace('0', 'Unknown')
    
    # Removing hidden spaces (strip) and unifying capitalization (upper)
    text_columns = ['InvoiceNo', 'StockCode', 'Description', 'Country']
    for col in text_columns:
        df[col] = df[col].astype(str).str.strip().str.upper()
    
    # Ensuring these are numeric to prevent errors during financial reconciliation
    df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce').fillna(0).astype(int)
    df['UnitPrice'] = pd.to_numeric(df['UnitPrice'], errors='coerce').fillna(0)
    
    return df

df_oms = clean_and_standardize(df_oms)
df_wms = clean_and_standardize(df_wms)

print("Standardization complete. Data types and formatting are now aligned.")

Standardization complete. Data types and formatting are now aligned.


**Step 5** Duplicates detection

In [26]:
# identify all occurrences of duplicates (including the originals)
# keep=False is used so that we capture every row that has a twin for audit purposes
wms_duplicates_audit = df_wms[df_wms.duplicated(keep=False)]
oms_duplicates_audit = df_oms[df_oms.duplicated(keep=False)]

#move duplicated records into separated files in the validation folder
wms_duplicates_audit.to_csv('validation/wms_duplicates_audit.csv', index=False)
oms_duplicates_audit.to_csv('validation/oms_duplicates_audit.csv', index=False)

# create the final "clean" DataFrames by dropping duplicates, here we keep only the first occurrence of each record
df_oms_final = df_oms.drop_duplicates()
df_wms_final = df_wms.drop_duplicates()

# final Verification of row counts
print(f"Validation stage finished")
print(f"Duplicates moved to: validation/wms_duplicates_audit.csv")
print(f"Clean OMS records: {len(df_oms_final)}")
print(f"Clean WMS records: {len(df_wms_final)}")

# Final check: are the files equal in size now?
difference = len(df_oms_final) - len(df_wms_final)
if difference == 0:
    print("Dataframes are perfectly aligned by row count!")
else:
    print(f"There is still a difference of {abs(difference)} records.")
    
    
df_oms_final.to_csv('validation/oms_final_clean.csv', index=False)
df_wms_final.to_csv('validation/wms_final_clean.csv', index=False)
print(f"Main processing files: validation/oms_final_clean.csv and wms_final_clean.csv")
print(f"Audit files created: wms_duplicates_audit.csv, oms_duplicates_audit.csv")

Validation stage finished
Duplicates moved to: validation/wms_duplicates_audit.csv
Clean OMS records: 536641
Clean WMS records: 536659
There is still a difference of 18 records.
Main processing files: validation/oms_final_clean.csv and wms_final_clean.csv
Audit files created: wms_duplicates_audit.csv, oms_duplicates_audit.csv


**Conclusions**

Missing data verification:
- OMS: 1,454 missing descriptions, 135,080 missing CustomerIDs
- WMS: 1,454 missing descriptions, 134,660 missing CustomerIDs, 1,000 missing UnitPrices
- Missing UnitPrices in WMS will require special handling during financial reconciliation

Data type and  format standardization:

- Converted InvoiceDate to datetime format; identified 11 corrupted date entries in WMS (ERR_DATE_2024)
- Standardized CustomerID: converted from mixed float/string to consistent string format, removed hidden spaces
- Unified text fields (InvoiceNo, StockCode, Description, Country) to uppercase and stripped whitespace
- Ensured Quantity and UnitPrice are numeric types

Technical Data Cleaning

- Removed 11 date errors from WMS (moved to audit file)
- Identified and isolated duplicate records:
- OMS: 5,268 duplicate rows
- WMS: 5,702 duplicate rows (434 more than OMS)
- Created deduplicated "clean" datasets for comparison

Final Dataset Status

- OMS final clean: 536,641 records
- WMS final clean: 536,659 records
- Remaining discrepancy: 18 records (to be investigated in Comparison phase)

Audit Files Created:

wms_date_errors_audit.csv - 11 records with corrupted dates
oms_duplicates_audit.csv - 5,268 duplicate records
wms_duplicates_audit.csv - 5,702 duplicate records